## Why?
- to transfer data
- No-Cloning Theorem: can't just copy a quantum state; will get collapsed doing so

## How?
- Take 2 entangled qubits (A1, A2)
- A third qubit has the info to send (B)
- A1 and B are entangled by one of the two persons
- A1, B both measured, that measurement (cbits) sent to person having A2
- todo


In [1]:
!pip3 install -q qiskit qiskit-aer qiskit[visualization] pylatexenc pennylane pennylane-qiskit

In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

### Defining qubits:
- 0: B
- 1: A1
- 2: A2

In [3]:
qc = QuantumCircuit(3, 2) #2 cbits for B and A1

### Setting up B: (Eg: Minus state here)

In [4]:
qc.x(0) #randomly applying some gate to make it secret interesting instead of 0
qc.h(0)

### Setting up A1 entangled with A2:

In [5]:
qc.h(1)
qc.cx(1, 2)
qc.barrier() #line sep in draw

CircuitInstruction(operation=Instruction(name='barrier', num_qubits=3, num_clbits=0, params=[]), qubits=(<Qubit register=(3, "q"), index=0>, <Qubit register=(3, "q"), index=1>, <Qubit register=(3, "q"), index=2>), clbits=())

### Entangling A1 with B:

In [6]:
qc.cx(0, 1) #Checking flip (0/1)
qc.h(0) #Phase (+/-) checking with first qbit
qc.barrier()

CircuitInstruction(operation=Instruction(name='barrier', num_qubits=3, num_clbits=0, params=[]), qubits=(<Qubit register=(3, "q"), index=0>, <Qubit register=(3, "q"), index=1>, <Qubit register=(3, "q"), index=2>), clbits=())

### Why qc.h(0) here?
- measure() cannot output the phase (sign) of qubit
- qc.h(0) converts the phase to another bit
- When measured:
  - 00: +, no flip
  - 01: +, flip
  - 10: -, no flip
  - 11: -, flip

### Measuring A1, B; sending the cbits to A2 over the internet (or however):

In [7]:
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()

CircuitInstruction(operation=Instruction(name='barrier', num_qubits=3, num_clbits=0, params=[]), qubits=(<Qubit register=(3, "q"), index=0>, <Qubit register=(3, "q"), index=1>, <Qubit register=(3, "q"), index=2>), clbits=())

### Reconstructing B (the message) by Person #2 using A2:

In [8]:
with qc.if_test((qc.clbits[1], 1)):
  qc.x(2)
with qc.if_test((qc.clbits[0], 1)):
  qc.z(2)

- z gate: phase change (+ to -, - to +)
- c_if(X, Y): apply if cbitX has value Y
  - x gate (flip 1/0) if 1 at cbit1 (which stores flip)
  - z gate (flip +/-) if 1 at cbit0 (which stores phase)

In [9]:
print(qc.draw())

     ┌───┐┌───┐ ░      ┌───┐ ░ ┌─┐    ░                                      »
q_0: ┤ X ├┤ H ├─░───■──┤ H ├─░─┤M├────░──────────────────────────────────────»
     ├───┤└───┘ ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐ ░                                      »
q_1: ┤ H ├──■───░─┤ X ├──────░──╫─┤M├─░──────────────────────────────────────»
     └───┘┌─┴─┐ ░ └───┘      ░  ║ └╥┘ ░   ┌──────  ┌───┐ ───────┐   ┌──────  »
q_2: ─────┤ X ├─░────────────░──╫──╫──░───┤ If-0  ─┤ X ├  End-0 ├───┤ If-0  ─»
          └───┘ ░            ░  ║  ║  ░   └──╥───  └───┘ ───────┘   └──╥───  »
                                ║  ║    ┌────╨────┐               ┌────╨────┐»
c: 2/═══════════════════════════╩══╩════╡ c_1=0x1 ╞═══════════════╡ c_0=0x1 ╞»
                                0  1    └─────────┘               └─────────┘»
«                    
«q_0: ───────────────
«                    
«q_1: ───────────────
«     ┌───┐ ───────┐ 
«q_2: ┤ Z ├  End-0 ├─
«     └───┘ ───────┘ 
«c: 2/═══════════════
«                    
